# 🟡 Olist CDC Pipeline
## Databricks SA Interview Prep — CDC & Delta Lake Capstone
---
**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Estimated time:** 2–3 hours  
**Difficulty:** Intermediate → Advanced

### What you'll build
A full CDC-based ingestion pipeline simulating a real-world migration from nightly full table dumps to incremental Delta Lake updates — modelled on a conversation you'd have with a Databricks customer.

```
Olist Postgres (simulated)
        │
        ▼
[CDC Simulator]       ← Debezium-style INSERT / UPDATE / DELETE events
        │
        ▼
[Bronze]              ← raw CDC envelope, full fidelity, no transforms
        │
        ├─────────────────────────────────┐
        ▼                                 ▼
[Silver: Current State]       [SCD Type 2 History]
  one row per order_id          full audit trail with effective dates
        │
        ▼
[Gold: Daily Order Summary]   ← incremental aggregation via signed deltas
```

### Milestones
| # | Name | Key concepts |
|---|------|-------------|
| 0 | Setup & Config | SparkSession, Delta, schema definition |
| 1 | Bronze Ingest | Streaming JSON, CDC envelope, Delta append |
| 2 | Silver Current State | Delta MERGE, deduplication, soft deletes |
| 3 | SCD Type 2 History | Effective dating, Delta time travel |
| 4 | Multi-Table CDC | Generalised MERGE handler, schema evolution |
| 5 | Gold Aggregations | Incremental deltas, signed MERGE |

> **How to use this notebook:** each milestone has explanation cells, then stub cells with `# TODO` markers. Fill in the TODOs. Interview questions are at the end of each milestone — answer them out loud before moving on.

---
## Milestone 0 — Setup & Configuration

Before writing any pipeline code, get your environment and paths in order.  
All tunable constants live here — change them once, they flow everywhere.

In [1]:
import os

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'

In [4]:
# Create all local pipeline directories (run once per fresh notebook session).
for p in PIPELINE_PATHS:
    os.makedirs(p, exist_ok=True)

print("✅ Pipeline directories ready")
for p in PIPELINE_PATHS:
    print("  -", p)

✅ Pipeline directories ready
  - /Users/gdoan/code/sandbox/Olist/data/olist/cdc_input
  - /Users/gdoan/code/sandbox/Olist/data/olist/bronze/cdc_events
  - /Users/gdoan/code/sandbox/Olist/data/olist/silver/orders
  - /Users/gdoan/code/sandbox/Olist/data/olist/silver/products
  - /Users/gdoan/code/sandbox/Olist/data/olist/history/orders_scd2
  - /Users/gdoan/code/sandbox/Olist/data/olist/gold/order_summary
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/silver
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/history
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/gold


In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Stabilize local Spark networking on macOS hostnames.
os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

# If a SparkSession already exists (without Delta jars), stop it first.
# Otherwise getOrCreate() can silently reuse a non-Delta session.
try:
    spark.stop()
except Exception:
    pass

builder = (
    SparkSession.builder
    .appName("OlistCDC")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Quick sanity checks so failures surface early.
print("✅ SparkSession ready — Spark", spark.version)
print("   spark.sql.extensions:", spark.conf.get("spark.sql.extensions", "<unset>"))
print("   spark.sql.catalog.spark_catalog:", spark.conf.get("spark.sql.catalog.spark_catalog", "<unset>"))
print("   spark.jars.packages:", spark.conf.get("spark.jars.packages", "<unset>"))

:: loading settings :: url = jar:file:/Users/gdoan/code/sandbox/Olist/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/gdoan/.ivy2.5.2/cache
The jars for the packages stored in: /Users/gdoan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59c29dcc-5f5f-400c-82cd-2a8f06b51fd2;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.2.0 in central
	found org.roaringbitmap

✅ SparkSession ready — Spark 4.1.1
   spark.sql.extensions: io.delta.sql.DeltaSparkSessionExtension
   spark.sql.catalog.spark_catalog: org.apache.spark.sql.delta.catalog.DeltaCatalog
   spark.jars.packages: io.delta:delta-spark_4.1_2.13:4.2.0


In [6]:
# Cleanup helper: remove all generated CDC/Delta/checkpoint folders for a truly fresh run.
import shutil

for p in PIPELINE_PATHS:
    shutil.rmtree(p, ignore_errors=True)

print("🧹 Removed pipeline folders")
for p in PIPELINE_PATHS:
    print("  -", p)
print("Re-run the directory setup cell before starting streams again.")

🧹 Removed pipeline folders
  - /Users/gdoan/code/sandbox/Olist/data/olist/cdc_input
  - /Users/gdoan/code/sandbox/Olist/data/olist/bronze/cdc_events
  - /Users/gdoan/code/sandbox/Olist/data/olist/silver/orders
  - /Users/gdoan/code/sandbox/Olist/data/olist/silver/products
  - /Users/gdoan/code/sandbox/Olist/data/olist/history/orders_scd2
  - /Users/gdoan/code/sandbox/Olist/data/olist/gold/order_summary
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/silver
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/history
  - /Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/gold
Re-run the directory setup cell before starting streams again.


## Schema & Constants
### Schema Definitions

Define all schemas explicitly here. Three layers to think about:

1. **CDC envelope schema** — the Debezium wrapper (`op`, `ts_ms`, `before`, `after`)
2. **Order payload schema** — the actual order row nested inside `before`/`after`
3. **Silver schema** — what your cleaned current-state table looks like

> **Design question before you write anything:** should `before` and `after` in the envelope be `StringType` (raw JSON) or a proper nested `StructType`? What are the trade-offs of each approach?

In [7]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType,
    TimestampType, BooleanType
)

# ── Order payload — matches olist_orders_dataset.csv columns ─────────────────
ORDER_PAYLOAD_SCHEMA = StructType([
    StructField("order_id",                      StringType(),  True),
    StructField("customer_id",                   StringType(),  True),
    StructField("order_status",                  StringType(),  True),
    StructField("order_purchase_timestamp",      StringType(),  True),  # cast downstream
    StructField("order_approved_at",             StringType(),  True),
    StructField("order_delivered_carrier_date",  StringType(),  True),
    StructField("order_delivered_customer_date", StringType(),  True),
    StructField("order_estimated_delivery_date", StringType(),  True),
])

# ── CDC envelope — Debezium-style wrapper ─────────────────────────────────────
CDC_ENVELOPE_SCHEMA = StructType([
    StructField("op",     StringType(),  False),  # I / U / D
    StructField("ts_ms",  LongType(),    False),  # source event time (epoch ms)
    StructField("before", ORDER_PAYLOAD_SCHEMA, True),
    StructField("after",  ORDER_PAYLOAD_SCHEMA, True),
    StructField("source", StructType([
        StructField("table", StringType(), True),
        StructField("db",    StringType(), True),
    ]), True),
])

print("✅ Schemas defined")
# print(CDC_ENVELOPE_SCHEMA.simpleString())  # uncomment to inspect

✅ Schemas defined


In [18]:
import os

# -- Data paths -----------------------------------------------------------------
# Drop your Kaggle CSVs in the same directory as this notebook under data/raw/
BASE_DIR     = os.getcwd()
DATA_DIR     = os.path.join(BASE_DIR, "data", "raw")

ORDERS_CSV   = os.path.join(DATA_DIR, "olist_orders_dataset.csv")
PRODUCTS_CSV = os.path.join(DATA_DIR, "olist_products_dataset.csv")

# -- CDC stream input/output ----------------------------------------------------
# Keep all generated lakehouse artifacts under data/olist/ for easy local cleanup.
PIPELINE_DIR   = os.path.join(BASE_DIR, "data", "olist")
CDC_INPUT_DIR  = os.path.join(PIPELINE_DIR, "cdc_input")

BRONZE_PATH    = os.path.join(PIPELINE_DIR, "bronze")
SILVER_ORDERS  = os.path.join(PIPELINE_DIR, "silver", "orders")
SILVER_PRODUCTS = os.path.join(PIPELINE_DIR, "silver", "products")
HISTORY_ORDERS = os.path.join(PIPELINE_DIR, "history")
GOLD_SUMMARY   = os.path.join(PIPELINE_DIR, "gold", "order_summary")

# -- Checkpoints (one per streaming query - never share between queries) --------
CKPT_BRONZE    = os.path.join(PIPELINE_DIR, "checkpoints", "bronze")
CKPT_SILVER    = os.path.join(PIPELINE_DIR, "checkpoints", "silver")
CKPT_HISTORY   = os.path.join(PIPELINE_DIR, "checkpoints", "history")
CKPT_GOLD      = os.path.join(PIPELINE_DIR, "checkpoints", "gold")

# -- Spark tuning ---------------------------------------------------------------
SHUFFLE_PARTITIONS    = 8      # keep low for local; raise on a cluster
TRIGGER_INTERVAL      = "10 seconds"
MAX_FILES_PER_TRIGGER = 1

# -- Simulator settings ---------------------------------------------------------
CDC_BATCH_SIZE      = 100
CDC_INTERVAL_SECS   = 3.0
CDC_LIMIT_ROWS      = 50_000

# Centralized list used by setup/cleanup helper cells below.
PIPELINE_PATHS = [
    CDC_INPUT_DIR,
    BRONZE_PATH,
    SILVER_ORDERS,
    SILVER_PRODUCTS,
    HISTORY_ORDERS,
    GOLD_SUMMARY,
    CKPT_BRONZE,
    CKPT_SILVER,
    CKPT_HISTORY,
    CKPT_GOLD,
]

print("✅ Config loaded")
print(f"   Orders CSV   : {ORDERS_CSV}")
print(f"   Pipeline dir : {PIPELINE_DIR}")
print(f"   CDC input    : {CDC_INPUT_DIR}")
print(f"   Bronze       : {BRONZE_PATH}")

✅ Config loaded
   Orders CSV   : /Users/gdoan/code/sandbox/Olist/data/raw/olist_orders_dataset.csv
   Pipeline dir : /Users/gdoan/code/sandbox/Olist/data/olist
   CDC input    : /Users/gdoan/code/sandbox/Olist/data/olist/cdc_input
   Bronze       : /Users/gdoan/code/sandbox/Olist/data/olist/bronze


---
## CDC Simulator
### Provided — no implementation needed

This simulates an operational Postgres database emitting row-level changes  
in **Debezium format** — the industry-standard CDC envelope used by tools  
like Debezium, Fivetran, and Airbyte.

Debezium is an open source CDC tool that captures row level changes then streams them elsewhere usually kafka
While most database use their WAL/transaction log to recover state during failures, Debezium uses it to transmit row level event logs. This allows us to read changes as they happen in real time
Each event looks like:
```json
{
  "op":     "U",
  "ts_ms":  1704067200000,
  "before": { "order_id": "abc", "order_status": "shipped", ... },
  "after":  { "order_id": "abc", "order_status": "delivered", ... },
  "source": { "table": "orders", "db": "olist" }
}
```

> **Note:** The simulator intentionally **shuffles events** before writing them.  
> This simulates out-of-order CDC delivery — a real problem you'll need to solve  
> in Milestone 2. Pay attention to how this affects your MERGE logic.

In [19]:
import json, os, random, time, threading
from datetime import datetime
import pandas as pd

CDC_INSERT = "I" # insert
CDC_UPDATE = "U" # update
CDC_DELETE = "D" # delete

STATUS_PROGRESSION = {
    "created":    "approved",
    "approved":   "processing",
    "processing": "shipped",
    "shipped":    "delivered",
}

def simulate_cdc_stream(
    orders_csv=ORDERS_CSV,
    output_dir=CDC_INPUT_DIR,
    batch_size=CDC_BATCH_SIZE,
    interval_seconds=CDC_INTERVAL_SECS,
    limit_rows=CDC_LIMIT_ROWS,
    verbose=False,
):
    def log(msg: str) -> None:
        if verbose:
            print(msg)

    os.makedirs(output_dir, exist_ok=True)
    log(f"[simulator] reading {limit_rows:,} rows...")
    df = pd.read_csv(orders_csv, nrows=limit_rows)
    df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

    order_state, events = {}, []

    for _, row in df.iterrows():
        order_id = row["order_id"]
        row_dict  = row.to_dict()
        initial   = {**row_dict, "order_status": "created"}

        events.append({
            "op": CDC_INSERT, "ts_ms": int(datetime.now().timestamp() * 1000),
            "before": None, "after": initial,
            "source": {"table": "orders", "db": "olist"},
        })
        order_state[order_id] = initial

        current = "created"
        for _ in range(random.randint(1, 3)):
            nxt = STATUS_PROGRESSION.get(current)
            if not nxt:
                break
            before = dict(order_state[order_id])
            after  = {**before, "order_status": nxt}
            events.append({
                "op": CDC_UPDATE, "ts_ms": int(datetime.now().timestamp() * 1000),
                "before": before, "after": after,
                "source": {"table": "orders", "db": "olist"},
            })
            order_state[order_id] = after
            current = nxt

    random.shuffle(events)   # intentional out-of-order delivery
    total = len(events) // batch_size
    log(f"[simulator] writing {len(events):,} events in {total} batches to {output_dir}")

    for i in range(total):
        batch = events[i * batch_size : (i + 1) * batch_size]
        path  = os.path.join(output_dir, f"cdc_{i:05d}.json")
        with open(path, "w") as f:
            for e in batch:
                f.write(json.dumps(e) + "\n")
        # log(f"[simulator] batch {i+1}/{total} -> {path}")
        time.sleep(interval_seconds)
    log("[simulator] complete.")

print("✅ Simulator defined — run the next cell to start it")

✅ Simulator defined — run the next cell to start it


In [20]:
# Start the simulator in a background thread
# Run this BEFORE starting your Bronze streaming query

SIMULATOR_VERBOSE = False  # set True if you want per-batch logs in notebook output

simulator_thread = threading.Thread(
    target=simulate_cdc_stream,
    kwargs={"verbose": SIMULATOR_VERBOSE},
    daemon=True,   # dies automatically when the notebook kernel stops
)
simulator_thread.start()
print("✅ Simulator running in background")
print(f"   Writing to: {CDC_INPUT_DIR}")
print(f"   Verbose logs: {SIMULATOR_VERBOSE}")

✅ Simulator running in background
   Writing to: /Users/gdoan/code/sandbox/Olist/data/olist/cdc_input
   Verbose logs: False


---
## Milestone 1 — Bronze: Raw CDC Event Landing
**Concepts:** streaming JSON ingest, CDC envelope schema, Delta append, checkpointing

### Goal
Read the CDC event stream from `CDC_INPUT_DIR` and land every event —  
inserts, updates, and deletes — to a Bronze Delta table with **zero transformation**.  
Preserve the full Debezium envelope including `op`, `ts_ms`, `before`, and `after`.

### Design decision (answer this before writing code)
> Should Bronze store the full Debezium envelope, or just the `after` image?  
> Write your reasoning as a comment in the implementation cell below.  
> This is a decision you'll be asked to defend in an interview.

In an effort to preserve the full events as they happen and are recievved, we should take the entire envelope as it comes in.  This provides a raw form of data of the full events before transformations occur. Having this bronze table with events as they come in lets us re-derive silver/gold tables if we need to fix logic of some sort or we want to create new tables for analytics.  By storing after only we lose the lineage of rows if they are deleted or updated since all we see is the row after it is changed. This does come at the expense of increased cost of storage, but its benefits in replayability and auditability is an important gain we have from this cost.

In [26]:
# MILESTONE 1 IMPLEMENTATION
# ─────────────────────────────────────────────────────────────────────────────

# read CDC_INPUT_DIR as a streaming JSON source

cdc_stream = spark.readStream.format('json').schema(CDC_ENVELOPE_SCHEMA).option("maxFilesPerTrigger", MAX_FILES_PER_TRIGGER).load(CDC_INPUT_DIR)

# verify it's actually a streaming DataFrame before writing anything
print(cdc_stream.isStreaming)
print(cdc_stream.schema)

# write to BRONZE_PATH as Delta append with checkpointing and trigger
query_bronze = (
    cdc_stream
    .writeStream \
    .format("delta")
    .outputMode("append")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .option("checkpointLocation", CKPT_BRONZE)
    .start(BRONZE_PATH)
)

True
StructType([StructField('op', StringType(), True), StructField('ts_ms', LongType(), True), StructField('before', StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', StringType(), True), StructField('order_approved_at', StringType(), True), StructField('order_delivered_carrier_date', StringType(), True), StructField('order_delivered_customer_date', StringType(), True), StructField('order_estimated_delivery_date', StringType(), True)]), True), StructField('after', StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', StringType(), True), StructField('order_approved_at', StringType(), True), StructField('order_delivered_carrier_date', StringType(), True), StructField('order_delivered_customer_date', StringT

26/06/15 21:11:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/15 21:11:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [22]:
!find {CKPT_BRONZE} -type f | head -20

/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/.metadata.crc
/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/sources/0/.0.crc
/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/sources/0/0
/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/offsets/.0.crc
/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/offsets/0
/Users/gdoan/code/sandbox/Olist/data/olist/checkpoints/bronze/metadata


In [27]:
# Verify Bronze is working — run after a few batches have landed
# Stop with query_bronze.stop() when done
query_bronze.stop()
# read Bronze back as a static Delta table and inspect it
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.groupBy("op").count().show()

# What does the checkpoint directory contain?
# !find {CKPT_BRONZE} -type f | head -20
# Outputs the directory of the chekcpoint location showing the commits, metadata and .crc file
# the .crc file is an integrity checker created by hadoop/spark

26/06/15 21:12:28 WARN DAGScheduler: Failed to cancel job group 7acc7789-e0e7-4047-aed3-d0c7617ad384. Cannot find active jobs for it.
26/06/15 21:12:28 WARN DAGScheduler: Failed to cancel job group 7acc7789-e0e7-4047-aed3-d0c7617ad384. Cannot find active jobs for it.


Bronze row count: 600
+---+-----+
| op|count|
+---+-----+
|  I|  193|
|  U|  407|
+---+-----+



### ✅ Milestone 1 — Interview Questions
Answer these out loud before moving to Milestone 2:

1. Why preserve the full envelope (`op`, `before`, `after`) rather than just the `after` image?


    We preserve the full envelope for the bronze table because we want the full state of changes that happened in the database.  By having the full envelope, we can replay events as they occured in the database, re-derive/derive new silver and gold tables as needed, and have a way to recover and audit database transactions with the entire state of the transaction including what changed and what is new.  For UPDATE cases, it provides insight on what has changed in the database, capturing the entire old state to new state transaction


2. How is Bronze in this CDC pipeline different from Bronze in a streaming append pipeline (like the taxi project)?
    
    The CDC pipeline processes all mutations of state in a database where as the taxi bronze only captures and appends new events (rides as they happen). Therefore instead of capturing only new facts (taxi pipeline), this pipeline captures new facts and changes to old facts.  Both of these define the events that occur.  Where the taxi pipeline is 1 record is 1 departure and 1 arrival, the CDC pipeline is 1 record either a U | D | I.


3. What does `ts_ms` represent — and how does it differ from the time the file arrived in `CDC_INPUT_DIR`?
    

    ts_ms represents the time at which the transaction occurred on the database.  Due to the nature of distributed systems, it is likely that there will be some lag between the occurrence of the event and the time it arrives at the input directory.  In order to accurately capture the time at which the event occurred for transformations or accurately encapsulating a timeline, we use ts_ms. The consequence of not using ts_ms could be having inaccurate timeline of events which would corrupt watermarks, joins, and windowed aggregations


4. What does the checkpoint directory contain after 5 batches? What would happen if you deleted it and restarted?
    

    after 5 batches, the checkpoint directory will contains a WAL and offset files.  These 2 files in their directories represent a "checkpoint" for the engine to use to figure out what was the last transaction processed which helps the spark engine prevent recomputing data that is already there.  If the directory is deleted and restarted, the engine act as though it has never seen data in the source before and write everything to the destination location as if it were new meaning there is a risk of duplicates in the sink. The WAL represents the "what i plan to write" log for the engine where as the offset updates only after the write is completed.  So we can identify the gap of where the engine may have failed.  The WAL is what is behind the exactly-once semantics of delta storage


---
## Milestone 2 — Silver: Current-State Orders Table
**Concepts:** Delta MERGE, within-batch deduplication, soft deletes, idempotency

### Goal
Maintain a **current-state table** — one row per `order_id`, always reflecting  
the latest known state — by applying CDC events from Bronze using Delta MERGE.

### The deduplication problem (solve this on paper first)
Because the simulator shuffles events, a single micro-batch may contain:
- An `INSERT` and two `UPDATE` events for the same `order_id`
- Multiple `UPDATE` events for the same `order_id` in different status stages

Applying them in arrival order will corrupt your Silver table.

> **Before writing any code:** what column tells you which event is truly the latest?  
> Is that column reliable? What edge cases exist?
     
     
     The latest timestamp contains the most accurate status of the order_id.  Therefore as we process records for de-duplication we would want to order it by ts_ms as opposed to the time the file lands because files can land in the destination sink out of order due to the nature of distributed operations taking time to travel over the network.  Furthermore, if we are processing in micro-batches, there is a chance the data lands at the same time, so the time the data lands doesnt always capture the correct ordering of events. 
     
     Lets consider the event where 2 records come in with the same ts_ms and different operations.  
    
     In the case of an Insert and Delete having the ts_ms, we need an additional tiebreaker which encapusulates perfect final state.  Well lets note that an insert must appear before a delete and an update must appear before a delete.  In this case if we're filter on rank=1 and by the last ts_ms (desc), we want to make sure that the D always wins.  so the order by would be ts_ms desc, op asc

### MERGE semantics
```
WHEN MATCHED AND op = 'U'  →  update all columns + cdc_updated_at
WHEN MATCHED AND op = 'D'  →  set is_deleted = true (soft delete)
WHEN NOT MATCHED           →  insert new row with is_deleted = false
```

In [28]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame, Window, Row
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
import time

def resolve_batch(batch_df: DataFrame) -> DataFrame:
    """
    Given a micro-batch of CDC events that may contain multiple events
    per order_id, return a deduplicated DataFrame with one row per order_id
    representing the correct final state to apply.

    implement deduplication logic.

    Steps to think through:
      1. What column determines which event is 'latest' for a given order_id?
          ts_ms desc -> always the latest event
          op asc -> prioritize deletes in the case of ts_ms ties
      2. How do you handle DELETE events — their 'after' field is null?
          we can mark them for soft delete with an additional field called "to-delete" then let some cron job run through and delete rows
      3. Use a window function to rank events per order_id and keep the top one.

      select ...
        rank() over partition by order_id order by ts_ms desc, op desc
      group by order_id
    """
    # NOTE: partitionBy uses the plain column name "order_id" — NOT "source.order_id".
    # The "source" alias is applied at MERGE time, not here. Using "source.order_id"
    # here treats the dot as a literal column name, which Spark cannot resolve.
    w = Window.partitionBy("order_id").orderBy(F.desc("ts_ms"), F.asc("op"))
    return batch_df.withColumn("rank", F.rank().over(w)).filter("rank == 1")

def apply_cdc_merge(micro_batch_df: DataFrame, batch_id: int) -> None:
    """
    foreachBatch handler: apply resolved CDC events to the Silver table.

      1. Call resolve_batch to deduplicate the micro-batch
      2. Separate into upserts (op I or U) and deletes (op D)
      3. For upserts: MERGE into SILVER_ORDERS
           WHEN MATCHED    → updateAll + set cdc_updated_at
           WHEN NOT MATCHED → insertAll with is_deleted=false
      4. For deletes: MERGE into SILVER_ORDERS
           WHEN MATCHED → set is_deleted=true, cdc_updated_at=now()

    This function must be IDEMPOTENT.
    Running it twice on the same batch_id must produce the same result.
    How do you guarantee that?
    """
    payload = F.coalesce(F.col("after"), F.col("before"))

    deduped_resolved = resolve_batch(micro_batch_df.select(
        F.col("op"),
        F.col("ts_ms"),
        payload.getField("order_id").alias("order_id"),
        payload.getField("customer_id").alias("customer_id"),
        payload.getField("order_status").alias("order_status"),
        payload.getField("order_purchase_timestamp").alias("order_purchase_timestamp"),
        payload.getField("order_approved_at").alias("order_approved_at"),
        payload.getField("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        payload.getField("order_delivered_customer_date").alias("order_delivered_customer_date"),
        payload.getField("order_estimated_delivery_date").alias("order_estimated_delivery_date"),
        F.when(F.col("op") == "D", F.lit(True)).otherwise(F.lit(False)).alias("is_deleted"),
        F.current_timestamp().alias("cdc_updated_at"),
    ))

    spark = SparkSession.getActiveSession()
    if spark:
        DeltaTable.forPath(spark, SILVER_ORDERS).alias("target") \
            .merge(deduped_resolved.alias("source"), "target.order_id = source.order_id") \
            .whenMatchedUpdate(
                condition="source.op = 'U'",
                set={
                    "order_status":                  "source.order_status",
                    "order_approved_at":             "source.order_approved_at",
                    "order_delivered_carrier_date":  "source.order_delivered_carrier_date",
                    "order_delivered_customer_date": "source.order_delivered_customer_date",
                    "order_estimated_delivery_date": "source.order_estimated_delivery_date",
                    "cdc_updated_at":                "source.cdc_updated_at",
                }
            ) \
            .whenMatchedUpdate(
                condition="source.op = 'D'",
                set={
                    "is_deleted":     "source.is_deleted",
                    "cdc_updated_at": "source.cdc_updated_at",
                }
            ) \
            .whenNotMatchedInsert(
                condition="source.op != 'D'",
                values={
                    "order_id":                      "source.order_id",
                    "customer_id":                   "source.customer_id",
                    "order_status":                  "source.order_status",
                    "order_purchase_timestamp":      "source.order_purchase_timestamp",
                    "order_approved_at":             "source.order_approved_at",
                    "order_delivered_carrier_date":  "source.order_delivered_carrier_date",
                    "order_delivered_customer_date": "source.order_delivered_customer_date",
                    "order_estimated_delivery_date": "source.order_estimated_delivery_date",
                    "is_deleted":                    "source.is_deleted",
                    "cdc_updated_at":                "source.cdc_updated_at",
                }
            ) \
            .execute()

In [29]:
(DeltaTable.createIfNotExists(spark) \
    .addColumns(ORDER_PAYLOAD_SCHEMA) \
    .addColumn("is_deleted", BooleanType(), nullable=True) \
    .addColumn("cdc_updated_at", TimestampType(), nullable=True) \
    .location(SILVER_ORDERS) \
    .execute())

In [ ]:
# Stop Bronze query first if still running
# TODO: read BRONZE_PATH as a Delta stream
bronze_stream = spark.readStream \
.format("delta") \
.option("maxFilesPerTrigger", MAX_FILES_PER_TRIGGER) \
.load(BRONZE_PATH)

# Start Silver query
query_silver = (
    bronze_stream
    .writeStream
    .foreachBatch(apply_cdc_merge)
    .option("checkpointLocation", CKPT_SILVER)
    .option("mergeSchema", "true")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

26/06/15 21:13:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/15 21:13:12 WARN MapPartitionsRDD: RDD 197 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 21:13:15 WARN MapPartitionsRDD: RDD 272 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [31]:
# Verify Silver — run after a few batches

silver_df = spark.read.format("delta").load(SILVER_ORDERS)
print(f"Silver row count (unique orders): {silver_df.count()}")
print(f"Deleted orders: {silver_df.filter('is_deleted = true').count()}")
print()
silver_df.groupBy("order_status").count().orderBy("count", ascending=False).show()

# Spot check: find an order that has gone through multiple status transitions
# and verify Silver shows only its latest state

Silver row count (unique orders): 199
Deleted orders: 0

+------------+-----+
|order_status|count|
+------------+-----+
|     created|   66|
|    approved|   56|
|  processing|   48|
|     shipped|   29|
+------------+-----+



26/06/15 21:13:22 WARN MapPartitionsRDD: RDD 376 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


### ✅ Milestone 2 — Interview Questions

1. How did you solve the within-batch deduplication problem? What column did you use and why?


    ts_ms is the timestamp column to use and in order to break ties, we prioritize deletes as they signal the row has been removed from the database.  We use ts_ms over landing time because ts is tied to when the event actually occurred while landing time has noticeable room for error because of network latency, and we let db deletes rule over other operations as they indicate the row is no longer in the table. The within-batch deduplication leverages a window over dropDuplicates in order to emphasize deterministic behavior: a delete should always rank first over any other behavior


2. Why soft-delete (`is_deleted = true`) rather than physically removing rows from a lakehouse?


    When a row is marked to be softly deleted, it only has a column marking for deletion as opposed to the write job deleting the row altogether.  This allows for an external job, potentially on a cron schedule, to go in and batch delete all rows marked for soft deletes.  The reason this is likely better to due on its own schedule is because the responsibility of physically removing a row has an additional component of needing to physically remove and reorganize the table post deletion which introduces potential latency in writes.  Delegating this responsibility to an external worker allows the writer to only be focused on marking tables with their most up-to-date values. Furthermore, having an ad-hoc job externally manage physical deletion allows for better replayability between when a row is physically removed from database memory as it can keep track of all data that has been deleted in an operation and replay if necessary.

    From the perspective of delta time travel, rows marked with deletion still exist in every part of the table.  Time travel then lets us see at which version of the table a certain row was marked to be deleted.  The soft deletion also provides protection against Vacuum cleaning up unused unreferenced rows
    
    Downstream consumers who are responsible for auditing or compliance trying to consume records from odler versions of the table would not be able to access hard-deleted records. 


3. What makes your MERGE function idempotent? How would you prove it?

    
    The merge function written in the code above is idempotent because of its overrwrite semantics and condition handling.  In both conditions (matched & not matched), we allow the entire row to be re-written resulting in duplicate batches ending with the same state: the 'after' in the input record. If this were to be doing a stateful aggregation, however, we would need to pay more attention to idempotency as accumulation semantics cannot guarantee the same state after processing without additional logic to handle repeat batches. We can prove it by replaying a single batch that updates a row twice and seeing that row in the silver table having matching values to its state before the record.  Row count and dataframe equality are both valid options for comparison.


4. What happens if a DELETE event arrives in a batch *before* its corresponding INSERT has been processed?

    In our case, the delete event would be skipped because it matches none of the conditions in the merge logic. The insert row is processed, and the row is inserted to the silver table resulting in an inconsistent state with the database.  

    you've identified this is a correctness bug. What would you do about it? There are a few different approaches depending on how much you're willing to complicate the pipeline — think about what information you'd need to detect and fix this case.

    we can fail on the delete at the expense of shutting down the entire pipeline, which would be bad, so probably not that.  I think if a delete is recieved on an absent row, we could give it to a queue orsomething like that to be retried as we await the inserted row to come but that would probably invite some ifnrastructural overhead and thinking aboyut things like queue outages etc.  I think we can also insert the record as marked for deletion and no lnoger let the inserted row into the database.  this approach leaves deleted rows as the supreme , so when the insert record is procsesed it is practically dropped. I think thats probably the best approach here. That approach is called inserting as a tombstone

---
## Milestone 3 — SCD Type 2: Full Order History
**Concepts:** Slowly Changing Dimensions, effective dating, Delta time travel

### Goal
The current-state Silver table tells you what every order looks like *now*.  
The compliance team needs to answer: **"What was the status of order X at 2pm yesterday?"**

Build a history table using **SCD Type 2** logic. Every status change:
- Closes the old row (`effective_to = now, is_current = false`)
- Opens a new row (`effective_from = now, effective_to = null, is_current = true`)

### Target schema
```
order_id        string     natural key
order_status    string     value for this version  
effective_from  timestamp  when this version became active
effective_to    timestamp  when superseded (null = still current)
is_current      boolean    true for the active version only
cdc_ts_ms       long       source event timestamp
```

### Then verify with Delta time travel
After building the history table, use:
```python
spark.read.format("delta").option("timestampAsOf", "2024-01-15 14:00:00").load(SILVER_ORDERS)
```
Compare the time-travel result against what your SCD2 table says was current at that moment.

In [32]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, LongType
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()

HISTORY_SCHEMA = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("effective_from", TimestampType(), True),
    StructField("effective_to", TimestampType(), True),
    StructField("is_current", BooleanType(), True),
    StructField("cdc_ts_ms", LongType(), True),
])

if not DeltaTable.isDeltaTable(spark, HISTORY_ORDERS):
    spark.createDataFrame([], HISTORY_SCHEMA).write.format("delta").save(HISTORY_ORDERS)

26/06/15 21:13:33 WARN MapPartitionsRDD: RDD 436 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [ ]:
from pyspark.sql import DataFrame, functions as F, Window
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

def apply_scd2(micro_batch_df: DataFrame, batch_id: int) -> None:
    spark = SparkSession.getActiveSession()
    if spark is None:
        return

    history = DeltaTable.forPath(spark, HISTORY_ORDERS)

    # Keep only business-meaningful events for history:
    # - INSERTs
    # - UPDATEs where status actually changed
    candidates = (
        micro_batch_df
        .filter(
            ((F.col("op") == "I") & F.col("after").isNotNull()) |
            (
                (F.col("op") == "U") &
                F.col("before").isNotNull() &
                F.col("after").isNotNull() &
                (F.col("before.order_status") != F.col("after.order_status"))
            )
        )
        .select(
            F.col("after.order_id").alias("order_id"),
            F.col("after.order_status").alias("order_status"),
            F.col("ts_ms").alias("cdc_ts_ms"),
            F.col("op").alias("op"),
        )
    )

    # IMPORTANT: one event per order per micro-batch (latest wins),
    # so we don't create multiple "current" rows for the same order_id.
    w = Window.partitionBy("order_id").orderBy(F.desc("cdc_ts_ms"), F.desc("op"))
    latest_per_order = (
        candidates
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .withColumn("effective_from", (F.col("cdc_ts_ms") / 1000).cast("timestamp"))
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    # Pass 1: close existing current row for each order_id we are updating/inserting.
    history.alias("target").merge(
        latest_per_order.alias("source"),
        "target.order_id = source.order_id AND target.is_current = true"
    ).whenMatchedUpdate(
        set={
            "effective_to": "source.effective_from",
            "is_current": "false",
        }
    ).execute()

    # Pass 2: insert the new current version.
    history.alias("target").merge(
        latest_per_order.alias("source"),
        """
        target.order_id = source.order_id
        AND target.cdc_ts_ms = source.cdc_ts_ms
        AND target.order_status = source.order_status
        """
    ).whenNotMatchedInsert(
        values={
            "order_id": "source.order_id",
            "order_status": "source.order_status",
            "effective_from": "source.effective_from",
            "effective_to": "source.effective_to",
            "is_current": "source.is_current",
            "cdc_ts_ms": "source.cdc_ts_ms",
        }
    ).execute()

In [ ]:
# TODO: read Bronze as stream and start SCD2 query
bronze_stream_2 = spark.readStream.format("delta").option("maxFilesPerTrigger", 5).load(BRONZE_PATH)

query_history = (
    bronze_stream_2
    .writeStream
    .foreachBatch(apply_scd2)
    .option("checkpointLocation", CKPT_HISTORY)
    .trigger(processingTime="10 seconds")
    .start()
)

26/06/15 21:13:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/15 21:13:44 WARN MapPartitionsRDD: RDD 570 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 21:13:44 WARN MapPartitionsRDD: RDD 524 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 21:13:52 WARN MapPartitionsRDD: RDD 730 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 21:13:53 WARN MapPartitionsRDD: RDD 724 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [ ]:
from pyspark.sql import functions as F

def verify_with_time_travel(as_of_timestamp: str):
    as_of_col = F.to_timestamp(F.lit(as_of_timestamp), "yyyy-MM-dd HH:mm:ss")

    history_asof = (
        spark.read.format("delta").load(HISTORY_ORDERS)
        .filter(
            (F.col("effective_from") <= as_of_col) &
            ((F.col("effective_to") > as_of_col) | F.col("effective_to").isNull())
        )
        .select("order_id", "order_status")
        .dropDuplicates()
    )

    silver_asof = (
        spark.read.format("delta")
        .option("timestampAsOf", as_of_timestamp)
        .load(SILVER_ORDERS)
        .select("order_id", "order_status")
        .dropDuplicates()
    )

    only_in_history = history_asof.exceptAll(silver_asof)
    only_in_silver = silver_asof.exceptAll(history_asof)

    print("only_in_history:", only_in_history.count())
    print("only_in_silver :", only_in_silver.count())

    return only_in_history, only_in_silver

Total history rows: 12000
Current rows (is_current=True): 11304

Version history for order: 650ca3873d0a09fb4604deef7817ef60
+--------------------------------+------------+--------------------------+------------+----------+
|order_id                        |order_status|effective_from            |effective_to|is_current|
+--------------------------------+------------+--------------------------+------------+----------+
|650ca3873d0a09fb4604deef7817ef60|created     |2026-06-15 20:46:36.815108|NULL        |true      |
+--------------------------------+------------+--------------------------+------------+----------+



26/06/15 20:47:15 WARN MapPartitionsRDD: RDD 3764 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 20:47:20 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 10055 milliseconds
26/06/15 20:47:23 WARN MapPartitionsRDD: RDD 3937 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
26/06/15 20:47:33 WARN MapPartitionsRDD: RDD 4078 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [51]:
# ── Delta Time Travel verification ───────────────────────────────────────────
# Compare what SCD2 says was current at a given timestamp
# against what Delta time travel shows for Silver at that same timestamp

def verify_with_time_travel(as_of_timestamp: str):
    """
    TODO:
      1. Query HISTORY_ORDERS for rows where:
           effective_from <= as_of_timestamp AND
           (effective_to > as_of_timestamp OR effective_to IS NULL)
         → this is SCD2's view of the world at that timestamp

      2. Query SILVER_ORDERS using Delta time travel:
           spark.read.format("delta")
                .option("timestampAsOf", as_of_timestamp)
                .load(SILVER_ORDERS)
         → this is Delta's view at that timestamp

      3. Compare the two — how many rows agree? Any discrepancies?
    """
    condition = f"effective_from <= to_timestamp('{as_of_timestamp}', 'yyyy-MM-dd HH:mm:ss') AND (effective_to > to_timestamp('{as_of_timestamp}', 'yyyy-MM-dd HH:mm:ss') OR effective_to IS NULL)"
    history_df = spark.read.format("delta").load(HISTORY_ORDERS).filter(condition).select("order_id", "order_status")

    silver_df = spark.read.format("delta").option("timestampAsOf", as_of_timestamp).load(SILVER_ORDERS).select("order_id", "order_status")

    return history_df.exceptAll(silver_df)
  
if spark is not None:
  deltaTable = DeltaTable.forPath(spark, SILVER_ORDERS)
  latest = deltaTable.history().selectExpr("max(timestamp) as ts").first()["ts"]
  latest_str = latest.strftime("%Y-%m-%d %H:%M:%S")
  diff = verify_with_time_travel(latest_str)
  diff.show(truncate=False)

+--------------------------------+------------+
|order_id                        |order_status|
+--------------------------------+------------+
|b9fc3fbace8f69b00c2f461cd3422459|processing  |
|1b46c5cd349b4c1e7bda383bb8d64731|processing  |
|4560df0eb1f79fd95f8acef34281bfdb|created     |
+--------------------------------+------------+



### ✅ Milestone 3 — Interview Questions

1. What is the difference between SCD Type 1, Type 2, and Type 3? When would you recommend each?

    SCD stands for slowly changing dimension which is used to capture changes to tables in a database. It can be used to capture trends on a database from analyzing the changes that happen per row. SCD type 1 updates the row in place. We don't preserve any history of the row with this type of SCD. Use this when we don't need to preserve history which is generally more performant as the stream does not need to use resources to write elsewhere. We use type 1 when we don't need a history. No additional storage cost or need for additional columns for no auditable history. We are choosing to accept intentional overwrites. 

    SCD Type 2 tracks changes coming into rows in a seperate table that which lets us create a versioned history.  We have better auditability here as analysts can leverage a table that exists seperately from the database it is tracking changes for.  This comes at the expense of complexity when writing to a different location as well as the storage pressure of storing another potentially large table. Therefore, if a full auditable history is needed, we go with type 2

    SCD Type 3 tracks historical information of rows as seperate columns in the table. This is helpful when we want to track a small number of changes that happens to a row.  The tradeoff here is a full auditable history of a row for not having to rows and write to another table every time something changes. If we need a history that doesn't really need to track full history versions and we want to be concious about creating a table, we'll use type 3 and track changes as columns. Use for predictable and bounded change.

2. Delta time travel and SCD Type 2 both let you query historical state — how do they differ? What does each give you that the other doesn't?

    Delta history provides a specific set of columns? when implementing scd type 2 we can customize what goes into the table. Delta tables track table differences. SCD type 2 changes are row level changes. Delta has built in reversion capabilities that lets you revert tables as a whole that scd type 2 wont necessarily let you do. Delta Time Travell gives granualirity into controlling retention. You can set granularity but 

    Delta time travel gives you: table-level snapshots, built-in version retention, and one-command restore (RESTORE TABLE AS OF). Zero engineering overhead — it's automatic.
    SCD2 gives you: permanent business-level history (survives VACUUM), queryable with standard SQL by business key, portable across any query engine, and you control exactly which columns and changes are tracked.
    The punchline: Delta time travel is for ops and recovery; SCD2 is for business history and compliance. They're complementary, not substitutes.

3. Why is `is_current = true` a useful column even though you could derive it from `effective_to IS NULL`?

    We can partition the history table with an is_current = true meaning we are able to physically seperate rows into the active rows in the history.  This partition lets us skip rows that are not relevant to eh current table. Effective_to does not have the same operational strength at scale because nulls don't work as partition keys


4. Why does the SCD2 MERGE require two passes rather than a single MERGE statement?

    Merges can only produce at most a 1 row delta.  In the cases where there needs to be an updated row and the insertion of a new one, it must use 2 passes to close the row and insert the new. There is not built in mechanic for when matched update but also insert.  So 1 pass cannot yield a 2 row delta. 

---
## Milestone 4 — Multi-Table CDC Handler
**Concepts:** generalised MERGE, schema evolution, composite keys

### Goal
Extend the pipeline beyond orders. In production, a CDC pipeline ingests  
changes from dozens of tables simultaneously. Build a generalised handler  
that routes events by `source.table` and applies the correct MERGE per table.

### Two parts

**Part A — Write your own change generator for products**  
The simulator only covers orders. You write the products simulator:
- Read `PRODUCTS_CSV`
- Each batch: randomly select 5% of rows, emit `UPDATE` with ±10% price change
- Each batch: emit 2–3 synthetic `INSERT` events for new products
- Same Debezium envelope format as the orders simulator

**Part B — Generalised routing handler**  
Route Bronze events by `source.table` → correct MERGE logic + target table.

In [ ]:
# ── PART A: Write your own products CDC simulator ─────────────────────────────

def simulate_products_cdc(
    products_csv=PRODUCTS_CSV,
    output_dir=CDC_INPUT_DIR,
    batch_size=50,
    interval_seconds=CDC_INTERVAL_SECS,
    num_batches=20,
):
    """
    TODO: implement a CDC simulator for the products table.

    Each batch should contain a mix of:
      - UPDATE events: 5% of products with a ±10% price_change adjustment
        (add a 'price_change' field to the after image to simulate price updates)
      - INSERT events: 2-3 new synthetic product rows

    Use the same Debezium envelope format as simulate_cdc_stream.
    Write to the same CDC_INPUT_DIR — the Bronze layer handles all tables.

    Hint: products don't have a status lifecycle, so no progression logic needed.
    Think about what columns change realistically for a product over time.
    """
    # TODO
    ...

print("✅ Products simulator defined")

In [ ]:
# ── PART B: Generalised routing handler ──────────────────────────────────────

# Maps source.table → (target Delta path, primary key column)
TABLE_ROUTING = {
    "orders":   (SILVER_ORDERS, "order_id"),
    "products": (SILVER_PRODUCTS, "product_id"),
    # TODO: add customers if you want to extend further
}

def apply_merge_generic(
    micro_batch_df: DataFrame,
    target_path: str,
    key_col: str,
) -> None:
    """
    TODO: generalised MERGE handler parameterised on target_path and key_col.
    Same pattern as apply_cdc_merge in Milestone 2, but works for any table.

    Think about:
      - Does every table need soft deletes?
      - Does deduplication logic change per table?
      - How do you handle a table with a composite primary key?
    """
    # TODO
    ...


def route_and_apply(micro_batch_df: DataFrame, batch_id: int) -> None:
    """
    TODO: route CDC events to the correct handler.

    For each entry in TABLE_ROUTING:
      1. Filter micro_batch_df where source.table == table_name
      2. If non-empty, call apply_merge_generic with the right path and key
    """
    # TODO
    ...

In [ ]:
# TODO: start the multi-table routing query
# Note: this should read from Bronze (same source as Milestone 2)
# but use a DIFFERENT checkpoint location

# Think: can this query share a checkpoint with query_silver? Why or why not?

query_multi = (
    spark.readStream.format("delta").load(BRONZE_PATH)
    .writeStream
    .foreachBatch(route_and_apply)
    # TODO: .option("checkpointLocation", ...)
    # TODO: .trigger(...)
    .start()
)

### ✅ Milestone 4 — Interview Questions

1. How would you handle a source table with a **composite primary key** (e.g., `order_id + product_id`)?
2. What happens to your pipeline if the source adds a new column mid-stream? How would Delta's `mergeSchema` option help?
3. Should the multi-table routing query share a checkpoint with the Silver orders query? What would go wrong if it did?

---
## Milestone 5 — Gold: Incremental Aggregations
**Concepts:** signed delta pattern, incremental MERGE, avoiding full recompute

### Goal
Maintain a Gold table of **daily order counts by status**, updated incrementally  
as CDC events flow through — without recomputing from scratch on every batch.

### Target schema
```
order_date    date       date of order_purchase_timestamp
order_status  string     e.g. 'shipped', 'delivered', 'canceled'
order_count   long       number of orders in this date/status bucket
last_updated  timestamp  when this row was last touched
```

### The challenge: status transitions
When an order moves from `shipped` → `delivered`, you need to:
- **Decrement** the `shipped` count for that order's date
- **Increment** the `delivered` count for that order's date

The mental model: treat Gold as a **running sum** and each batch  
as a set of **signed increments** (`+1` / `-1`) to apply via MERGE.

```
UPDATE event (shipped → delivered):
  row 1: (order_date, 'shipped',   count_delta = -1)   ← subtract
  row 2: (order_date, 'delivered', count_delta = +1)   ← add
```

In [ ]:
def compute_deltas(micro_batch_df: DataFrame) -> DataFrame:
    """
    From a micro-batch of CDC events, produce signed increment rows.

    Output schema:
        order_date    date
        order_status  string
        count_delta   long     (positive = add, negative = subtract)

    TODO:
      1. Filter UPDATE events where order_status changed
         (before.order_status != after.order_status)
      2. For each: produce TWO rows
           (date, old_status, -1)   ← remove from old bucket
           (date, new_status, +1)   ← add to new bucket
      3. Filter INSERT events and produce:
           (date, status, +1)
      4. Union all rows, then groupBy(order_date, order_status).sum(count_delta)
         to collapse multiple events hitting the same bucket in one batch

    Hint: use F.to_date() to extract the date from the timestamp string.
    Hint: use F.lit(-1) and F.lit(1) for the signed counts.
    """
    # TODO
    ...


def apply_gold_merge(micro_batch_df: DataFrame, batch_id: int) -> None:
    """
    Apply signed deltas to the Gold table.

    TODO:
      1. Call compute_deltas to get increment rows
      2. If empty, return early
      3. MERGE into GOLD_SUMMARY on (order_date, order_status):
           WHEN MATCHED:
             order_count  = existing.order_count + delta.count_delta
             last_updated = current_timestamp()
           WHEN NOT MATCHED:
             insert (order_date, order_status, count_delta as order_count, now())

    Edge case: what if order_count goes negative due to out-of-order events?
    How would you guard against that?
    """
    # TODO
    ...

In [ ]:
# TODO: start Gold query
query_gold = (
    spark.readStream.format("delta").load(BRONZE_PATH)
    .writeStream
    .foreachBatch(apply_gold_merge)
    .option("checkpointLocation", CKPT_GOLD)
    # TODO: trigger
    .start()
)

In [ ]:
# Verify Gold — inspect the aggregation table

gold_df = spark.read.format("delta").load(GOLD_SUMMARY)
print(f"Gold rows: {gold_df.count()}")
print()
gold_df.orderBy("order_date", "order_status").show(20, truncate=False)

### ✅ Milestone 5 — Interview Questions

1. Why is a full recompute from Bronze on every batch unacceptable at scale?
2. How does this incremental aggregation pattern differ from the windowed streaming aggregations in the taxi project?
3. What are the consistency guarantees when a downstream dashboard reads Gold while a MERGE is in flight?
4. What happens if `count_delta` causes `order_count` to go negative? When would that happen and how would you prevent it?

---
## Stretch Goals

Work through these if you finish early or want to push further.

**1. Exactly-once with batch_id logging**  
Write processed `batch_id` values to a Delta log table after each successful  
`foreachBatch` execution. On restart, skip any `batch_id` already in the log.  
This gives you true idempotency across restarts independent of checkpointing.

**2. Late-arriving deletes**  
Modify the simulator to emit a DELETE event for a random order 10 batches  
after its INSERT was already processed. Verify your Silver table handles it  
correctly without corrupting state. What does the history table show?

**3. Schema evolution mid-stream**  
Add a new field `estimated_delivery_date` to the simulator's order payload  
after the stream has been running for 10 batches. Handle it gracefully using  
Delta's `mergeSchema` option. What happens without it?

**4. Observability layer**  
In a `foreachBatch` wrapper, log per-batch metrics to a Delta table:
- `batch_id`, `batch_timestamp`
- `events_received`, `inserts`, `updates`, `deletes`
- `processing_duration_ms`

This is what a production observability layer looks like.

---
## The SA Walkthrough

When you're done, practice walking an interviewer through this pipeline in **10 minutes**.  
Cover these five points — this is the structure of a strong SA presentation round answer:

1. **The business problem**  
   Why were nightly full dumps failing the customer? What specifically was broken  
   (latency, cost, lost history, compliance)?

2. **The MERGE pattern**  
   Why is MERGE the right primitive for CDC? What makes it idempotent?  
   What would break if you used append-only writes instead?

3. **SCD Type 2 vs Delta time travel**  
   When would you recommend each to a customer? They solve overlapping  
   but distinct problems — articulate the difference clearly.

4. **What breaks first at scale**  
   High-velocity tables with thousands of updates per second hitting a single  
   MERGE — what degrades? How would you fix it?  
   (Think: batching strategies, partitioning, write amplification)

5. **The migration story**  
   A customer is running Glue + Redshift full dumps today.  
   Walk me through how you'd migrate them to this architecture.  
   What are the risks? What do you de-risk first?

> The last point is your SA differentiator.  
> Anyone can build the pipeline. The SA guides the customer through the transition.